## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.
This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.
## TODAY:
- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path
from huggingface_hub import InferenceClient
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from sklearn.manifold import TSNE
import plotly.graph_objects as go

C:\Users\user\AppData\Local\Temp\ipykernel_14692\1857798744.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


# dividee documents into chunks 
* if we convert a whole document into vector it is much less likely.to match a question with entire document
* user most likely ask a qustion which matches some part of doc
* by dividing into chunks we will increase the chances of matching it
## one more thing
* divideing into chunks is fully a trial and error method
* we cannot say this division is best
* we need keep on find best

In [22]:
model='gpt-4.1'

In [ ]:
# taking all the data from files and storing in one variable
# checking how many characters it has
knowledge_path="knowledge-base/**/*.md"
files=glob.glob(knowledge_path,recursive=True)
print(f"no of files are:{len(files)}")
entire_knowledge_base=""
for file in files:
    with open(file,'r',encoding="utf-8")as f:
        entire_knowledge_base+=f.read()
        entire_knowledge_base+="\n\n"
print(f"total no of character:{len(entire_knowledge_base)}")

no of files are:76
total no of character:304434


In [24]:
# how many tokens it has
encoding=tiktoken.encoding_for_model(model)
tokens=encoding.encode(entire_knowledge_base)
print(f"no of tokens are:{len(tokens)}")

no of tokens are:63555


In [29]:
# load everything in knowledge base using langchain and turn them into langchain object called 'document'
folders=glob.glob('knowledge-base/*')
documents=[]
for folder in folders:# iterating each folder
    doc_type=os.path.basename(folder)#folder name will be doctype like emplyee,products...
    loader=DirectoryLoader(folder,glob='**/*.md',loader_cls=TextLoader,loader_kwargs={"encoding":"utf-8"})
    folder_docs=loader.load()
    for doc in folder_docs:
        doc.metadata['doc_type']=doc_type # giving name to the metadata[doctype=will be folder name]
        documents.append(doc)
print(f"length of documents:{len(documents)}")

length of documents:76


In [35]:
# now each document is loaded in langchain document format
# every document is loaded in documents
documents[3]

Document(metadata={'source': 'knowledge-base\\company\\overview.md', 'doc_type': 'company'}, page_content="# Overview of Insurellm\n\nInsurellm is an innovative insurance tech firm with 32 employees operating primarily remotely across the US, with offices in San Francisco (HQ), New York, Austin, Chicago, and Denver.\n\nFounded in 2015, the company has evolved from a high-growth startup to a lean, profitable operation focused on sustainable growth and operational excellence.\n\n## Products\n\nInsurellm offers 8 insurance software products across multiple insurance lines:\n\n### Core Insurance Portals\n- **Carllm** - Auto insurance platform for insurers\n- **Homellm** - Home insurance platform for insurers\n- **Lifellm** - Life insurance platform with AI-powered underwriting\n- **Healthllm** - Comprehensive health insurance platform\n- **Bizllm** - Commercial insurance platform for business coverage\n\n### Marketplace & Infrastructure\n- **Markellm** - Marketplace connecting consumers wi

# now we split the data into chunks 
* divideing into chunks is fully a trial and error method
* we cannot say this division is best
* we need keep on find best
* we use text splitters
# now we using recursive character splitters
* first tries to splits text werever it finds couple of empty lines
* next by single empty line 
* and then by end of sentence
* it has this heirarchy to divide the sentence where it naturally ends

# chunk size=how many characters you want roughly
* chunk_overlap=how many characters overlap between different chunks
* Chunk overlap means repeating a small part of one chunk in the next chunk.
* same text in two chunks
* Chunk overlap = repeating some text between consecutive chunks to preserve context.(then cances will be less for RAG to miss context)


# we also have markdowntextsplitter
### which can be used now bcoz our files are md

In [41]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=text_splitter.split_documents(documents)

print(f"divided into {len(chunks)} chunks")
# we should find best chunk strategy that is most essential aspects for rag workflow
# we will  cover that later this week

divided into 413 chunks


In [40]:
print(chunks[20])

page_content='2. **Pricing Review**: Renewal pricing will be discussed 75 days prior to term end and mutually agreed upon in writing. Price increases limited to 7% annually.

## Features

1. **Included Features**: Advantage Medical Coverage will have access to Healthllm Professional Tier features:
   - Intelligent Plan Design with competitive benchmarking
   - Real-Time Eligibility Verification with provider integration
   - AI-Driven Claims Adjudication with auto-processing
   - Predictive Healthcare Analytics for population health
   - Provider Network Management with 8,000+ providers
   - Member Engagement Platform with mobile app
   - Advanced Medication Management with PBM integration
   - Regulatory Compliance Engine (ACA, state mandates, HIPAA)
   - Enhanced Analytics and predictive modeling
   - Care management tools and workflows' metadata={'source': 'knowledge-base\\contracts\\Contract with Advantage Medical Coverage for Healthllm.md', 'doc_type': 'contracts'}


# now we turn this documents chunks into vectors
* remember encoder turns text into vectors
* vector datastore is database where we store vectors. which quickly finds the nearest point for the question vector
* vector datastore we use here is chroma
* Vector stores are databases designed to store and quickly search vector embeddings based on similarity.

In [46]:
db_name = "vector_db"

In [ ]:
embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
# selected the embedding model which converts text into vectors

if os.path.exists(db_name):
    Chroma(persist_directory=db_name,embedding_function=embeddings).delete_collection()
    # it creates a database folder every time we run this it will delete all data 
vectorstore=Chroma.from_documents(documents=chunks,embedding=embeddings,persist_directory=db_name)# this embedd chunks and store
# documents and embedding models instance is passed and directory where vectors to be stored
print(f"vectorstore created with {vectorstore._collection.count()}documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

vectorstore created with 413documents


# if we use open ai paid model
## embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
* it has 3072 dimensions
## embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
* it has 1536 dimensions
* we select the model which better capture the meaning of text
* model with higher dimension has better freedom of expressing meaning

In [49]:
# lets see a vector
collection=vectorstore._collection
count=collection.count()

sample_vector=collection.get(limit=1,include=['embeddings'])["embeddings"][0]
dimensions=len(sample_vector)# dimension of each vector
print(f"there are {count}vectors with {dimensions} dimensions")

there are 413vectors with 384 dimensions


# visualizing vectors

In [67]:
result=collection.get(include=['embeddings','documents','metadatas'])
vectors=np.array(result['embeddings'])
documents=result['documents']
metadatas=result['metadatas']
doc_type=[metadata['doc_type']for metadata in metadatas]
colors=[['blue','green','red','orange'][['products','employees','contracts','company'].index(t)]for t in doc_type]

# text=text=[f"type:{t},Text:{d[:100]}..."for t,d in zip[Any(doc_type,documents)]]
* This creates the information that appears when you hover over a point.
* d[:100] first 100 characters of document
* <br> next line in html
* this text is given to hover info
* which shows each dot info when we hover on that

In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)
tsne=TSNE(n_components=2,random_state=42)# 2 means 2d,384 dimensions to 2
reduced_vectors=tsne.fit_transform(vectors)# passing embeddings 

# Create the 2D scatter plot
fig=go.Figure(data=[go.Scatter(
    x=reduced_vectors[:,0],# take 0th column
    y=reduced_vectors[:,1],# take 1st column
    mode='markers',# Tells Plotly to display the data as dots/points
    marker=dict(size=5,color=colors,opacity=0.8),# size of dot,colors of dots,opacity
    text=[f"type:{t},Text:{d[:100]}..."for t,d in zip(doc_type,documents)],
    hoverinfo='text'
)])
fig.update_layout(
    title="2d chroma vector store visualization",
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)# This controls the empty space (margin) around your Plotly graph r-right margin...

)
fig.show()

# lets be clear that 
* encoder makes these 384 dimensions not vector datastore it only stores

# here we dont see what is x axis what is y axis just seee distrubution vectors
* x and y aaxis doesnot have any meaning
* vectors which are close have similar properties
* which are far have less similar properties

# 3d visualisation

In [71]:
tsne=TSNE(n_components=3,random_state=42)
reduced_vectors=tsne.fit_transform(vectors)

fig=go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:,0],
    y=reduced_vectors[:,1],
    z=reduced_vectors[:,2],
    mode='markers',
    marker=dict(size=5,color=colors,opacity=0.8),
    text=[f"type:{t},Text:{d[:100]}.."for t,d in zip(doc_type,documents)],
    hoverinfo='text'
)])
fig.update_layout(
    title="3D chrome vectors representation",
    scene=dict(xaxis_title='x',yaxis_title='y',zaxis_title='z'),
    width=800,
    height=600,
    margin=dict(r=20,b=10,l=20,t=40)
)
fig.show()